<a href="https://colab.research.google.com/github/MohinaRustamova/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohinaRustamova/lyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My target is a yes/no observed label (is_declining_label), so the toolkit points at
Logistic Regression first, then Random Forest. I already have a grouped-split
Logistic Regression from ML-05 (Precision@20 = 0.350, Precision@50 = 0.500 on 12
honest features) so this week I formalize that as the real model, then add a
Random Forest to see if it beats it. Lane 2 cares about ranking, not classification
accuracy, so both models get scored by Precision@K, not accuracy or AUC. The real
comparison this week isn't model-vs-naive-baseline, it's model-vs-my-Week-4-rule,
on the same held-out rows.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ---- Setup: load data and rebuild feature vector (matches ML-05 pattern) ----
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL, REPO_DIR = "https://github.com/MohinaRustamova/flyrank-ml-internship", "flyrank-ml-internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

%pip -q install duckdb

import duckdb
import pandas as pd
import numpy as np

con = duckdb.connect()

if IN_COLAB:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
else:
    from getpass import getpass
    HF_TOKEN = getpass("HF_TOKEN: ")

con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

FACT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
DIMC = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"

DECISION_DATE = "2026-03-31"

# --- Feature vector, identical to ML-05 ---
feature_frame = con.sql(f"""
    WITH prior AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_prior30,
               SUM(gsc_clicks) AS clicks_prior30,
               AVG(gsc_avg_position) AS avg_position_prior30
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-01-31' AND DATE '2026-03-01'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    current AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_current30,
               SUM(gsc_clicks) AS clicks_current30
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT p.client_hash_id, p.content_hash_id,
           p.impressions_prior30, p.clicks_prior30, p.avg_position_prior30,
           c.impressions_current30, c.clicks_current30,
           CASE WHEN c.impressions_current30 < p.impressions_prior30 * 0.8
                THEN 1 ELSE 0 END AS is_declining_label
    FROM prior p
    JOIN current c USING (client_hash_id, content_hash_id)
    WHERE p.impressions_prior30 > 0
""").df()

feature_frame["ctr_prior30"] = (
    feature_frame["clicks_prior30"] / feature_frame["impressions_prior30"]
).replace([float("inf"), -float("inf")], 0).fillna(0)

feats = con.sql(f"""
    SELECT content_hash_id, content_type, word_count,
           DATE_DIFF('day', content_created_date, DATE '{DECISION_DATE}') AS content_age_days,
           last_optimized_date
    FROM {DIMC}
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""").df()

feature_frame = feature_frame.merge(feats, on="content_hash_id", how="left")

feature_frame["has_word_count"] = feature_frame["word_count"].notna().astype(int)
feature_frame["word_count"] = feature_frame["word_count"].fillna(0)

decision_ts = pd.to_datetime(DECISION_DATE)
last_opt = pd.to_datetime(feature_frame["last_optimized_date"])
known_optimization = last_opt.notna() & (last_opt <= decision_ts)

feature_frame["has_been_optimized"] = known_optimization.astype(int)
feature_frame["days_since_last_optimized"] = (decision_ts - last_opt).dt.days
feature_frame.loc[~known_optimization, "days_since_last_optimized"] = -1

feature_frame = pd.get_dummies(feature_frame, columns=["content_type"], prefix="type")

print(feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(135113, 18)


,client_hash_id,content_hash_id,impressions_prior30,clicks_prior30,avg_position_prior30,impressions_current30,clicks_current30,is_declining_label,ctr_prior30,word_count,content_age_days,last_optimized_date,has_word_count,has_been_optimized,days_since_last_optimized,type_comparison article,type_feedly article,type_keyword article
0,client_e547b89c05043229,content_e67934818ca184a1,320.0,1.0,23.767748,1072.0,2.0,0,0.003125,2905,351.0,2026-05-15,1,0,-1.0,False,False,True
1,client_e547b89c05043229,content_9634c35544bcc47b,411.0,0.0,11.136149,313.0,1.0,1,0.000000,2685,351.0,2026-05-27,1,0,-1.0,False,False,True
2,client_e547b89c05043229,content_4ece07fdea783709,629.0,1.0,10.486653,653.0,0.0,0,0.001590,0,351.0,NaT,0,0,-1.0,False,False,True
3,client_e547b89c05043229,content_4d9b25ca95147676,1421.0,13.0,5.133504,1028.0,18.0,1,0.009148,0,351.0,NaT,0,0,-1.0,False,False,True
4,client_e547b89c05043229,content_2dae25d4660d074a,1209.0,3.0,19.725096,1366.0,1.0,0,0.002481,2758,351.0,2026-05-27,1,0,-1.0,False,False,True


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


Grouped by client_hash_id, same split I validated in ML-05. A random split lets
rows from the same client sit in both train and test, so the model can partly
memorize per-client quirks instead of learning patterns that generalize to a
client it hasn't seen. In ML-05 the random split scored 0.047 higher than the
grouped split on accuracy alone, which is exactly that memorization gap. The
grouped split is the honest one, so it's the only one I'm using this week.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

honest_cols = [
    "impressions_prior30", "clicks_prior30", "ctr_prior30", "avg_position_prior30",
    "content_age_days", "days_since_last_optimized", "has_been_optimized",
    "word_count", "has_word_count",
    "type_comparison article", "type_feedly article", "type_keyword article",
]

X = feature_frame[honest_cols].fillna(0)
y = feature_frame["is_declining_label"]
groups = feature_frame["client_hash_id"]

base_rate = y.mean()
print(f"Base rate (share declining): {base_rate:.3f}")

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

# how many distinct clients ended up in each side, as a sanity check
print("Train clients:", groups.iloc[train_idx].nunique())
print("Test clients:", groups.iloc[test_idx].nunique())
print("Train rows:", len(Xtr), " Test rows:", len(Xte))

Base rate (share declining): 0.240
Train clients: 29
Test clients: 13
Train rows: 65018  Test rows: 70095


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I score my Week-4 rule (from w04_baseline_score.ipynb) on the same 42 test-set
content pieces the models never trained on, using the identical rule logic and
score formula. Then I train Logistic Regression and Random Forest on the
grouped-split train set and score all three on the same test rows, same metric:
Precision@20 and Precision@50. One table, one split, three rows.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# ---- Precision@K helper, reused for all three methods ----
def precision_at_k(labels_sorted_by_score_desc, k):
    return labels_sorted_by_score_desc.head(k).mean()

# ================================================================
# BASELINE: rebuild the Week-4 rule, restricted to test-set rows only
# ================================================================
# Reuses con, FACT, DIMC, DECISION_DATE from Cell 2 (Section 1).


baseline_df = con.sql(f"""
    WITH current_win AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS gsc_impressions,
               SUM(gsc_clicks) AS gsc_clicks,
               AVG(gsc_avg_position) AS gsc_avg_position
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-02' AND DATE '2026-03-31'
          AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    prior_win AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_prior30
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-01-31' AND DATE '2026-03-01'
          AND gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT c.content_hash_id,
           c.gsc_impressions, c.gsc_clicks, c.gsc_avg_position,
           p.impressions_prior30,
           CASE WHEN c.gsc_impressions < p.impressions_prior30 * 0.8
                THEN 1 ELSE 0 END AS is_declining_label
    FROM current_win c
    JOIN prior_win p USING (content_hash_id)
    WHERE p.impressions_prior30 > 0
""").df()

dim_content_age = con.sql(f"""
    SELECT content_hash_id, content_created_date FROM {DIMC}
""").df()
baseline_df = baseline_df.merge(dim_content_age, on="content_hash_id", how="left")

decision_ts = pd.to_datetime(DECISION_DATE)
baseline_df["content_created_date"] = pd.to_datetime(baseline_df["content_created_date"])
baseline_df["content_age_days"] = (decision_ts - baseline_df["content_created_date"]).dt.days

MIN_IMPRESSIONS = 100
AGING_BUCKETS = ["3-6mo", "6-12mo"]

age_bins = [0, 90, 180, 365, float("inf")]
age_labels = ["0-3mo", "3-6mo", "6-12mo", "12+mo"]
baseline_df["age_bucket"] = pd.cut(baseline_df["content_age_days"], bins=age_bins, labels=age_labels)

pos_bins = [0, 3, 6, 10, 20, float("inf")]
pos_labels = ["1-3", "3-6", "6-10", "10-20", "20+"]
baseline_df["position_bucket"] = pd.cut(baseline_df["gsc_avg_position"], bins=pos_bins, labels=pos_labels)
baseline_df["ctr"] = baseline_df["gsc_clicks"] / baseline_df["gsc_impressions"]

baseline_df["eligible"] = (
    (baseline_df["gsc_impressions"] >= MIN_IMPRESSIONS) &
    (baseline_df["position_bucket"] != "20+")
)

peer_ctr_map = (
    baseline_df[baseline_df["eligible"]]
    .groupby("position_bucket", observed=True)["ctr"]
    .median()
)
baseline_df["peer_median_ctr"] = baseline_df["position_bucket"].map(peer_ctr_map)
baseline_df["ctr_flag"] = baseline_df["eligible"] & (baseline_df["ctr"] < baseline_df["peer_median_ctr"] * 0.5)
baseline_df["aging_flag"] = baseline_df["age_bucket"].isin(AGING_BUCKETS)

expected_click_gap = (
    (baseline_df["peer_median_ctr"] - baseline_df["ctr"]).clip(lower=0) * baseline_df["gsc_impressions"]
).fillna(0)

baseline_df["score"] = (
    baseline_df["ctr_flag"].astype(int) * expected_click_gap * 2
    + baseline_df["aging_flag"].astype(int) * (baseline_df["gsc_impressions"] / 1000)
).round(1)

# --- Restrict to the SAME test-set rows the models are scored on ---
test_content_ids = feature_frame.iloc[test_idx]["content_hash_id"]
baseline_test = baseline_df[baseline_df["content_hash_id"].isin(test_content_ids)].copy()

# Sanity check: the rule's own label matches the model's label for these rows
label_check = baseline_test.merge(
    feature_frame.iloc[test_idx][["content_hash_id", "is_declining_label"]],
    on="content_hash_id", suffixes=("_rule", "_model")
)
mismatch = (label_check["is_declining_label_rule"] != label_check["is_declining_label_model"]).sum()
print(f"Test rows recovered for baseline: {len(baseline_test)} of {len(test_content_ids)}")
print(f"Label mismatches between rule and model (should be 0): {mismatch}")

baseline_ranked = baseline_test.sort_values("score", ascending=False)

# ================================================================
# MODELS: Logistic Regression and Random Forest, same train/test split
# ================================================================
scaler = StandardScaler()
Xtr_scaled = scaler.fit_transform(Xtr)
Xte_scaled = scaler.transform(Xte)

logreg = LogisticRegression(max_iter=2000).fit(Xtr_scaled, ytr)
logreg_probs = logreg.predict_proba(Xte_scaled)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1).fit(Xtr, ytr)
rf_probs = rf.predict_proba(Xte)[:, 1]

results = feature_frame.iloc[test_idx][["content_hash_id", "is_declining_label"]].copy()
results["logreg_prob"] = logreg_probs
results["rf_prob"] = rf_probs

logreg_ranked = results.sort_values("logreg_prob", ascending=False)
rf_ranked = results.sort_values("rf_prob", ascending=False)

# ================================================================
# COMPARISON TABLE
# ================================================================
rows = []
for name, ranked, label_col in [
    ("Baseline rule (Week 4)", baseline_ranked, "is_declining_label"),
    ("Logistic Regression", logreg_ranked, "is_declining_label"),
    ("Random Forest", rf_ranked, "is_declining_label"),
]:
    rows.append({
        "method": name,
        "precision@20": round(precision_at_k(ranked[label_col], 20), 3),
        "precision@50": round(precision_at_k(ranked[label_col], 50), 3),
    })

comparison = pd.DataFrame(rows)
comparison["base_rate"] = round(base_rate, 3)
print()
print(comparison.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Test rows recovered for baseline: 70095 of 70095
Label mismatches between rule and model (should be 0): 0

                method  precision@20  precision@50  base_rate
Baseline rule (Week 4)          0.10          0.12       0.24
   Logistic Regression          0.35          0.50       0.24
         Random Forest          0.30          0.26       0.24


In [10]:
# ---- Tie check: are rank-20 / rank-50 cutoffs sitting on a tie? ----

for name, ranked, prob_col in [
    ("Logistic Regression", logreg_ranked, "logreg_prob"),
    ("Random Forest", rf_ranked, "rf_prob"),
]:
    print(f"--- {name} ---")
    for k in [20, 50]:
        cutoff_score = ranked[prob_col].iloc[k - 1]  # score of the k-th ranked row
        tied_at_cutoff = (ranked[prob_col] == cutoff_score).sum()
        n_unique_scores = ranked[prob_col].nunique()
        print(f"  Rank {k}: score = {cutoff_score:.4f}, "
              f"rows sharing this exact score = {tied_at_cutoff}, "
              f"total unique scores in test set = {n_unique_scores}")
    print()

--- Logistic Regression ---
  Rank 20: score = 0.6156, rows sharing this exact score = 1, total unique scores in test set = 69758
  Rank 50: score = 0.5921, rows sharing this exact score = 1, total unique scores in test set = 69758

--- Random Forest ---
  Rank 20: score = 0.5996, rows sharing this exact score = 1, total unique scores in test set = 56978
  Rank 50: score = 0.5930, rows sharing this exact score = 1, total unique scores in test set = 56978



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Carrying Logistic Regression forward since it's more interpretable and more
consistent across queue depth (P@20 = 0.35, P@50 = 0.50) than Random Forest,
which wins narrowly at K=20 but drops off by K=50. I look at three things: which
features the model actually leans on (coefficients, sanity-checked against what
I know is and isn't leakage), where it's most wrong across position and age
groups, and three concrete misses with a reason for each.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ================================================================
# What the model leans on: coefficients (LogReg is linear, so these
# are directly readable — no permutation importance needed)
# ================================================================
coef_table = pd.DataFrame({
    "feature": honest_cols,
    "coefficient": logreg.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

print("Feature coefficients (sorted by absolute size):")
print(coef_table.to_string(index=False))

# ================================================================
# Where the model is most wrong: false negatives and false positives
# broken out by position bucket and age bucket, using the top-50
# predicted queue (the part of the ranking that's actually acted on)
# ================================================================
top50 = logreg_ranked.head(50).copy()
top50 = top50.merge(
    feature_frame.iloc[test_idx][["content_hash_id", "content_age_days"]],
    on="content_hash_id", how="left"
)
top50 = top50.merge(
    baseline_df[["content_hash_id", "position_bucket", "age_bucket"]],
    on="content_hash_id", how="left"
)

top50["correct"] = top50["is_declining_label"] == 1  # since these are all "predicted declining"

print("\nTop-50 predicted queue: correct vs wrong, by position bucket")
print(top50.groupby("position_bucket", observed=True)["correct"].agg(["count", "mean"]))

print("\nTop-50 predicted queue: correct vs wrong, by age bucket")
print(top50.groupby("age_bucket", observed=True)["correct"].agg(["count", "mean"]))

# ================================================================
# Three concrete wrong cases: highest-confidence false positives
# (model was very sure it was declining, but it wasn't)
# ================================================================
wrong_cases = top50[top50["correct"] == False].sort_values("logreg_prob", ascending=False).head(3)

other_cols = [c for c in honest_cols if c != "content_age_days"]
wrong_cases_full = wrong_cases.merge(
    feature_frame.iloc[test_idx][other_cols + ["content_hash_id"]],
    on="content_hash_id", how="left"
)

display_cols = ["content_hash_id", "logreg_prob", "content_age_days"] + other_cols
print("\nThree highest-confidence wrong picks:")
print(wrong_cases_full[display_cols].to_string(index=False))

Feature coefficients (sorted by absolute size):
                  feature  coefficient
  type_comparison article    -0.917465
     type_keyword article    -0.813773
               word_count     0.235941
           has_word_count    -0.220514
      type_feedly article    -0.217019
         content_age_days     0.108877
              ctr_prior30    -0.105824
     avg_position_prior30    -0.101030
      impressions_prior30    -0.062464
           clicks_prior30     0.044955
       has_been_optimized     0.000000
days_since_last_optimized     0.000000

Top-50 predicted queue: correct vs wrong, by position bucket
                 count      mean
position_bucket                 
3-6                  3  0.333333
6-10                40  0.450000
10-20                5  1.000000
20+                  1  0.000000

Top-50 predicted queue: correct vs wrong, by age bucket
            count      mean
age_bucket                 
0-3mo           3  0.333333
3-6mo           2  0.000000
6-12mo         4

Interpretation

**What the model leans on:** content type dominates. type_comparison article
(-0.92) and type_keyword article (-0.81) are by far the largest coefficients,
both negative, meaning those content types are predicted less likely to decline
than the reference category. word_count and has_word_count pull in opposite
directions (+0.24 vs -0.22), which is really one signal: longer content with
recorded word counts looks more stable. The prior-window performance features
(ctr_prior30, avg_position_prior30, impressions_prior30) all have small
coefficients, under 0.11 in magnitude, meaning recent performance carries less
weight than what the content IS. has_been_optimized and days_since_last_optimized
sit at exactly 0.0, which matches what I found in ML-04: no row in this dataset
had an optimization on or before the decision date, so the column is constant
here and the model correctly learned to ignore it rather than invent a signal.

**Where it's most wrong:** the top-50 predicted queue is worst in the 3-6mo age
bucket (0 of 2 correct) and best in 6-12mo (24 of 45 correct, 53%). Position-wise,
6-10 is the largest group in the top-50 (40 rows) and only 45% correct, close to
a coin flip despite the model being confident enough to rank these near the top.
Small buckets like 10-20 (5 rows, 100% correct) and 20+ (1 row, wrong) aren't
reliable enough to draw conclusions from given the sample size.

**Three concrete misses:**
- content_512dbad65bd5ade9 — 89% confidence, wrong. High prior-window volume
  (178K impressions) and a type_keyword article flag, both features the model
  weighs heavily. The model likely over-trusted the type signal here even
  though the underlying performance numbers looked fine.
- content_41edf11c713a82f3 — 87% confidence, wrong. Low volume (46 impressions,
  0 clicks) but flagged type_keyword article and never optimized. Thin data,
  the model may be leaning on content type as a substitute for real signal
  when performance history is sparse.
- content_578dcebdf98bff5b — 84% confidence, wrong. This one has content_age_days
  = NaN (missing content_created_date) and word_count = 0. Genuinely thin
  metadata, not just a hard case, this row probably should have been excluded
  or flagged rather than confidently scored.

**Overall:** the model leans more on what content IS (type, word count) than
on how it's recently performing, which is a different strategy than my Week-4
rule, which leaned entirely on recent CTR and age. That's likely why the model
beats the rule so clearly, it found a signal the rule never looked at.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.